# Step 4 · Candidate systems — Qwen3-8B

**Status:** One zero-shot Qwen experiment completed; other candidates remain planned.

Reproduce the Qwen experiment independently of the E5 notebook. Replay uses saved score files. This is a frozen LLM with development-label threshold tuning, not a fine-tuned model.

Start with a fresh runtime and execute cells in order. Read each section before running it. The original combined notebook and its outputs are retained as historical evidence.

## 1. Connect to the project
CPU is sufficient unless you explicitly enable new model inference. This uses the existing Drive project; it does not delete or recopy it.

In [ ]:
from pathlib import Path
import sys, os
try:
    from google.colab import drive
except ImportError:
    PROJECT = Path.cwd().resolve()
    if PROJECT.name == 'notebooks':
        PROJECT = PROJECT.parent
else:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/semeval2027/semeval2027_colab')
assert (PROJECT/'src/semeval_task3/data.py').is_file(), f'Edit PROJECT to your project root: {PROJECT}'
assert (PROJECT/'notebooks/project_utils.py').is_file(), 'Use the organized project, including notebooks/project_utils.py.'
sys.path.insert(0, str(PROJECT/'src'))
sys.path.insert(0, str(PROJECT/'notebooks'))
os.environ['PYTHONPATH'] = str(PROJECT/'src')
from project_utils import (run_checked, new_run, save_json, read_scores, save_scores,
                           frozen, standardized_by_word, record_environment, save_evaluation, digest)
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, word_bootstrap, read_submission
print('Project:', PROJECT)
print('Existing files stay in place. New outputs use unique directories under outputs/runs/.')

## 2. Configuration

In [ ]:
RUN_NEW_INFERENCE = False
MODEL_ID = 'Qwen/Qwen3-8B'
MODEL_REVISION = None
BATCH_SIZE = 8
run_checked([sys.executable,'-m','pip','install','--quiet','numpy'],cwd=PROJECT)
import numpy as np
dev = load_dataset(PROJECT/'work/data/dev',with_gold=True)
assert not any(u.span_flags for u in dev.usages), 'Resolve flagged spans before marked-target inference.'
RUN_DIR = new_run(PROJECT,'step04_qwen')
provenance = dict(mode='new_inference' if RUN_NEW_INFERENCE else 'legacy_score_replay',model=MODEL_ID)

## 3. Historical prompt
 We keep the original Swedish-specific wording for comparison. It is **not yet a six-language prompt**. Thinking is disabled; the score is the next-token Yes-versus-No logit difference. This is not a calibrated probability of correctness.

In [ ]:
SYSTEM_PROMPT = 'You are an expert lexicographer of historical Swedish.'
def messages_for(u):
    message = (f'Target word (lemma): {u.word}\n'
               f'Sense definition (Swedish): {u.definition}\n'
               f'Sentence from {u.year}: {u.marked_text()}\n\n'
               'Does the word marked with <t> and </t> in the sentence express the sense '
               'given by the definition? Answer only Yes or No.')
    return [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':message}]
print(messages_for(dev.usages[0]))
save_json(RUN_DIR/'prompt_example.json',messages_for(dev.usages[0]))

## 4. Load saved scores, or deliberately run a new GPU experiment

In [ ]:
import time
if not RUN_NEW_INFERENCE:
    cache = PROJECT/'outputs/qwen3_8b_zs_scores.json'
    qwen_scores = read_scores(cache,dev)
    provenance.update(source=str(cache.relative_to(PROJECT)),source_sha256=digest(cache),model_revision='unknown for historical run')
else:
    run_checked([sys.executable,'-m','pip','install','--quiet','transformers','accelerate'],cwd=PROJECT)
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from huggingface_hub import HfApi
    assert torch.cuda.is_available(), 'Select a GPU runtime before deliberately running inference.'
    revision = MODEL_REVISION or HfApi().model_info(MODEL_ID).sha
    tok = AutoTokenizer.from_pretrained(MODEL_ID,revision=revision)
    tok.padding_side = 'left'
    llm = AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=revision,dtype=torch.bfloat16,device_map='cuda').eval()
    yes_ids,no_ids = tok.encode('Yes',add_special_tokens=False),tok.encode('No',add_special_tokens=False)
    assert len(yes_ids)==len(no_ids)==1,(yes_ids,no_ids)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); start=time.perf_counter()
    qwen_scores = {}
    with torch.inference_mode():
        for i in range(0,len(dev.usages),BATCH_SIZE):
            batch = dev.usages[i:i+BATCH_SIZE]
            prompts = [tok.apply_chat_template(messages_for(u),tokenize=False,add_generation_prompt=True,enable_thinking=False) for u in batch]
            encoded = tok(prompts,return_tensors='pt',padding=True).to('cuda')
            logits = llm(**encoded).logits[:,-1,:].float()
            lp = torch.log_softmax(logits[:,[yes_ids[0],no_ids[0]]],dim=-1)
            qwen_scores.update({u.key:v for u,v in zip(batch,(lp[:,0]-lp[:,1]).tolist())})
            if i % 400 == 0:
                save_scores(RUN_DIR/'qwen_partial_scores.json',qwen_scores)
                print(i,'/',len(dev.usages))
    torch.cuda.synchronize()
    provenance.update(model_revision=revision,seconds=time.perf_counter()-start,gpu=torch.cuda.get_device_name(),
                      peak_gpu_gib=torch.cuda.max_memory_allocated()/1024**3,batch_size=BATCH_SIZE,dtype='bfloat16',thinking=False)
    del llm
    torch.cuda.empty_cache()
save_scores(RUN_DIR/'qwen3_8b_zs_scores.json',qwen_scores)
record_environment(PROJECT,RUN_DIR/'manifest.json',**provenance)
print('Raw Yes preference at zero threshold:',sum(v>0 for v in qwen_scores.values())/len(qwen_scores))

## 5. Nested threshold selection and comparison

In [ ]:
thresholds = tuple([-1000.0]+[round(x,2) for x in np.arange(-12,12.001,0.25)])
result = nested_word_cv(dev,[Candidate('qwen3_8b_zs',frozen(qwen_scores),thresholds)])
report = save_evaluation(RUN_DIR,'qwen3_8b_zs',result,dev,PROJECT)
save_json(RUN_DIR/'configuration.json',dict(thresholds=thresholds,system_prompt=SYSTEM_PROMPT,thinking=False,score='Yes minus No logit'))
for selection in result['selections']:
    print(selection)
e5_archive = PROJECT/'work/predictions/e5_raw.zip'
if e5_archive.exists():
    e5_report = score(dev.gold,read_submission(e5_archive,dev.keys))
    paired = word_bootstrap(result['report'],other=e5_report)
    save_json(RUN_DIR/'compare_qwen_vs_original_e5.json',paired)
    print('Difference vs original E5:',paired)
print('Saved:',RUN_DIR)

## Recorded result and remaining candidates
 Original Qwen F1 **0.462565**, below all-positive **0.471630**. Paired difference interval includes zero. Original runtime: about **154 seconds / 1,983 usages** on an L4; reported peak allocation **16.4 GiB**. These are historical measurements; a new run records its own values.

 **Pending:** prototype system (XL-LEXEME / XL-DURel), optional external-gloss cross-encoder, and any larger-model comparison. None has been implemented or declared a winner here.

 **Next:** Step 6 diagnostics. Broader model choices still require measured evidence.

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_fscore_support,
)

gold = dev.require_gold()

e5_run = (
    PROJECT / "outputs/runs/step03_baselines"
    / "20261103T163804Z_b602cbfb"
)

systems = {
    "Qwen": {
        "scores": qwen_scores,
        "predictions": read_submission(
            RUN_DIR / "qwen3_8b_zs.zip", dev.keys
        ),
    },
    "E5": {
        "scores": read_scores(PROJECT / "outputs/e5_raw_scores.json", dev),
        "predictions": read_submission(e5_run / "e5_raw.zip", dev.keys),
    },
}

diagnostics = []

for name, system in systems.items():
    print(f"\n{name} — held-out predictions")
    print(
        f"{'Word':<12} {'N':>4} {'Gold+%':>7} {'Pred+%':>7} "
        f"{'AUC':>6} {'Prec':>6} {'Recall':>6} {'F1':>6} {'ΔF1':>7}"
    )

    for group in sorted({u.group for u in dev.usages}):
        usages = [u for u in dev.usages if u.group == group]
        y = np.array([gold[u.key] for u in usages])
        s = np.array([system["scores"][u.key] for u in usages])
        pred = np.array([system["predictions"][u.key] for u in usages])
        assert np.isfinite(s).all()

        precision, recall, f1, _ = precision_recall_fscore_support(
            y, pred, average="binary", zero_division=0
        )
        auc = float(roc_auc_score(y, s)) if len(set(y)) == 2 else None
        positive_rate = float(y.mean())
        all_positive_f1 = 2 * positive_rate / (1 + positive_rate)

        row = {
            "system": name,
            "language": group[0],
            "word": group[1],
            "n": len(y),
            "gold_positive_rate": positive_rate,
            "predicted_positive_rate": float(pred.mean()),
            "auc": auc,
            "precision": float(precision),
            "recall": float(recall),
            "f1": float(f1),
            "all_positive_f1": all_positive_f1,
            "f1_difference": float(f1 - all_positive_f1),
        }
        diagnostics.append(row)

        auc_text = f"{auc:.3f}" if auc is not None else "N/A"
        print(
            f"{group[1]:<12} {len(y):>4} {100*positive_rate:>7.1f} "
            f"{100*pred.mean():>7.1f} {auc_text:>6} "
            f"{precision:>6.3f} {recall:>6.3f} {f1:>6.3f} "
            f"{f1-all_positive_f1:>+7.3f}"
        )

    valid_auc = [
        row["auc"] for row in diagnostics
        if row["system"] == name and row["auc"] is not None
    ]
    print(f"Mean per-word AUC: {np.mean(valid_auc):.4f}")

diagnostic_path = RUN_DIR / "qwen_e5_per_word_diagnostics.json"
save_json(diagnostic_path, {
    "rows": diagnostics,
    "note": (
        "Exploratory diagnostics on existing development predictions. "
        "No thresholds changed. AUC measures ranking; F1 uses held-out predictions."
    ),
})
print("\nSaved:", diagnostic_path)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

groups = sorted({u.group for u in dev.usages})
colors = {0: "#E69F00", 1: "#0072B2"}
distribution_rows = []

fig, axes = plt.subplots(
    1, len(systems), figsize=(15, 8), squeeze=False
)

for ax, (name, system) in zip(axes[0], systems.items()):
    print(f"\n{name} — score medians")
    print(f"{'Word':<12} {'Negative':>10} {'Positive':>10} {'Gap':>10}")

    for position, group in enumerate(groups):
        usages = [u for u in dev.usages if u.group == group]
        medians = {}

        for label, offset in [(0, -0.18), (1, 0.18)]:
            values = np.array([
                system["scores"][u.key]
                for u in usages if gold[u.key] == label
            ], dtype=float)

            if len(values) == 0:
                continue
            assert np.isfinite(values).all()

            p10, q1, median, q3, p90 = np.quantile(
                values, [0.10, 0.25, 0.50, 0.75, 0.90]
            )
            medians[label] = float(median)

            distribution_rows.append({
                "system": name,
                "language": group[0],
                "word": group[1],
                "label": label,
                "n": len(values),
                "p10": float(p10),
                "q25": float(q1),
                "median": float(median),
                "q75": float(q3),
                "p90": float(p90),
            })

            ax.bxp(
                [{
                    "whislo": float(p10),
                    "q1": float(q1),
                    "med": float(median),
                    "q3": float(q3),
                    "whishi": float(p90),
                    "fliers": [],
                }],
                positions=[position + offset],
                widths=0.28,
                vert=False,
                patch_artist=True,
                showfliers=False,
                manage_ticks=False,
                boxprops={"facecolor": colors[label], "alpha": 0.65},
                medianprops={"color": "black"},
            )

        if 0 in medians and 1 in medians:
            print(
                f"{group[1]:<12} {medians[0]:>10.4f} "
                f"{medians[1]:>10.4f} "
                f"{medians[1] - medians[0]:>+10.4f}"
            )

    ax.set_yticks(range(len(groups)))
    ax.set_yticklabels([group[1] for group in groups])
    ax.invert_yaxis()
    ax.set_title(name)
    ax.set_xlabel(
        "Yes − No logit" if name == "Qwen" else "Embedding cosine"
    )
    ax.grid(axis="x", alpha=0.25)
    ax.legend(handles=[
        Patch(facecolor=colors[0], label="Gold negative"),
        Patch(facecolor=colors[1], label="Gold positive"),
    ], loc="best")

fig.suptitle(
    "Swedish development score distributions\n"
    "Boxes: 25th–75th percentiles; whiskers: 10th–90th percentiles"
)
fig.tight_layout(rect=[0, 0, 1, 0.93])

plot_path = RUN_DIR / "score_distributions.png"
fig.savefig(plot_path, dpi=180, bbox_inches="tight")
save_json(RUN_DIR / "score_distributions.json", {
    "purpose": "Exploratory development diagnostics; no threshold fitting",
    "rows": distribution_rows,
})
plt.show()
plt.close(fig)

print("\nSaved plot:", plot_path)
print("Saved statistics:", RUN_DIR / "score_distributions.json")

In [ ]:
# Preserve the original Qwen evaluation for comparison.
original_qwen_report = score(
    dev.require_gold(),
    read_submission(RUN_DIR / "qwen3_8b_zs.zip", dev.keys),
)

qwen_z_scores = standardized_by_word(dev, qwen_scores)

# Fixed grids chosen before examining these experiments' results.
raw_grid = tuple(
    [-1000.0]
    + [round(float(x), 2) for x in np.arange(-40, 40.001, 0.25)]
    + [1000.0]
)
z_grid = tuple(
    [-1000.0]
    + [round(float(x), 2) for x in np.arange(-12, 12.001, 0.10)]
    + [1000.0]
)

# Stop rather than silently use a grid that misses the score range.
assert all(-40 <= v <= 40 for v in qwen_scores.values()), (
    "Raw scores exceed the proposed grid. Send me their min/max."
)
assert all(-12 <= v <= 12 for v in qwen_z_scores.values()), (
    "Standardized scores exceed the proposed grid. Send me their min/max."
)

CALIB_DIR = new_run(PROJECT, "step04_qwen_calibration")
save_scores(CALIB_DIR / "qwen_z_scores.json", qwen_z_scores)
save_json(CALIB_DIR / "configuration.json", {
    "raw_grid": list(raw_grid),
    "z_grid": list(z_grid),
    "normalization": "Per-word mean/std from unlabeled scores",
    "evaluation": "Separate nested word-CV evaluation for each variant",
    "original_qwen_run": str(RUN_DIR),
    "interpretation": (
        "Exploratory experiments motivated by development diagnostics; "
        "not independent confirmation on new data."
    ),
})

calibration_results = {}

for name, scores_to_use, grid in [
    ("qwen_raw_wide", qwen_scores, raw_grid),
    ("qwen_z", qwen_z_scores, z_grid),
]:
    print(f"\n--- {name} ---")
    evaluated = nested_word_cv(
        dev, [Candidate(name, frozen(scores_to_use), grid)]
    )
    save_evaluation(CALIB_DIR, name, evaluated, dev, PROJECT)

    comparison = word_bootstrap(
        evaluated["report"], other=original_qwen_report
    )
    save_json(CALIB_DIR / f"{name}_vs_original_qwen.json", comparison)

    print("Difference vs original Qwen:", comparison)
    print("Held-out thresholds:")
    for selection in evaluated["selections"]:
        print(
            selection["test_group"],
            "→", float(selection["threshold"])
        )

    calibration_results[name] = evaluated

record_environment(
    PROJECT, CALIB_DIR / "manifest.json",
    purpose="Qwen threshold-range and normalization experiments",
    source_scores_sha256=digest(RUN_DIR / "qwen3_8b_zs_scores.json"),
    new_model_inference=False,
)
print("\nSaved:", CALIB_DIR)

In [ ]:
import subprocess
import shutil
import psutil

subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.total,memory.free,driver_version",
    "--format=csv",
], check=True)

print(f"System RAM: {psutil.virtual_memory().total / 2**30:.1f} GiB")
print(f"Free disk: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")

In [ ]:
import subprocess

gpu_info = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv",
    ],
    capture_output=True,
    text=True,
    check=True,
)
print(gpu_info.stdout)

In [ ]:
import sys
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
assert (PROJECT / "notebooks/project_utils.py").is_file()

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        "transformers", "accelerate", "safetensors", "huggingface_hub",
    ],
    check=True,
)

import torch
import transformers
from huggingface_hub import HfApi
from transformers import AutoConfig, AutoProcessor, AutoModelForMultimodalLM

sys.path.insert(0, str(PROJECT / "notebooks"))
from project_utils import new_run, save_json, record_environment

assert torch.cuda.is_available(), "GPU is not connected."
assert torch.cuda.is_bf16_supported(), "BF16 support check failed."

# Check that this PyTorch build can execute on the Blackwell GPU.
with torch.inference_mode():
    probe = torch.randn(128, 128, device="cuda", dtype=torch.bfloat16)
    probe_result = probe @ probe
    assert torch.isfinite(probe_result).all().item()
del probe, probe_result
torch.cuda.empty_cache()

GEMMA_ID = "google/gemma-4-31B-it"
GEMMA_REVISION = HfApi().model_info(GEMMA_ID).sha
gemma_config = AutoConfig.from_pretrained(
    GEMMA_ID, revision=GEMMA_REVISION
)
assert gemma_config.model_type == "gemma4", gemma_config.model_type

GEMMA_RUN = new_run(PROJECT, "step04_gemma4_31b")
record_environment(
    PROJECT, GEMMA_RUN / "setup_manifest.json",
    model_id=GEMMA_ID,
    model_revision=GEMMA_REVISION,
    gpu=torch.cuda.get_device_name(0),
    planned_dtype="bfloat16",
    planned_thinking=False,
)

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("BF16 GPU operation: PASS")
print("Model configuration:", gemma_config.model_type)
print("Pinned revision:", GEMMA_REVISION)
print("Run directory:", GEMMA_RUN)